In [ ]:
import langchain, langchain_core, langchain_ollama
from langchain_ollama import ChatOllama              # Le LLM (notre "moteur")
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser  # Le parser de sortie
from pathlib import Path
import os

OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://host.docker.internal:11434")
llm = ChatOllama(
    model="gemma4:e4b",                  # Modèle local (à pré-télécharger : ollama pull gemma4:e4b)
    base_url=OLLAMA_URL,   # URL d'Ollama
    temperature=0                        # 0 = déterministe (idéal en formation)
)
 

def load_prompt(filepath: str) -> str:
    return Path(filepath).read_text(encoding="utf-8")

# Charger les templates depuis les fichiers
system_template = load_prompt("prompts/system_analyst.md")
user_template = load_prompt("prompts/user_query.md")

# Construire le ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(system_template),
    HumanMessagePromptTemplate.from_template(user_template),
])

# Utiliser le prompt
chain = prompt | llm

result = chain.invoke({
    "role": "data analyst",
    "years": "10",
    "language": "French",
    "domain": "finance",
    "user_input": "Analyse ces données de vente",
    "context": "Q1 2026, revenus en baisse de 5%",
})